In [25]:
import os, json
from markdown_it import MarkdownIt
from mdit_py_plugins.front_matter import front_matter_plugin
from mdit_py_plugins.footnote import footnote_plugin
from markdown_it.tree import SyntaxTreeNode
import re
import openai
from openai import OpenAI

Aux functions


In [26]:
def walk(node, depth=0):
    print("  " * depth, node.type)

    if hasattr(node, "children") and node.children:
        for child in node.children:
            walk(child, depth + 1)
#Load saved sessions in json
def load_ast_from_json(filepath = 'outputs/json_sessions.json'):
    try:
        with open(filepath, "r") as f:
            ast = json.load(f)
            return ast
    except json.JSONDecodeError as e:
        print("error decoding json from file")
    except FileNotFoundError as e:
        print(e)
    except Exception as e:
        print("Error loading ast")
        return None
def save_sessions(sessions,filepath = 'outputs/json_sessions.json') -> bool:
    json_sessions = json.dumps(sessions,indent=4)
    os.makedirs('outputs/',exist_ok=True)
    with open(filepath,"w") as f:
        f.write(json_sessions)

In [27]:
severity_titles = {
    "High Risk Findings": "high",
    "Medium Risk Findings": "medium",
    "Low Risk Findings": "low",
    "Non-Critical Findings": "non_critical",
    "Gas Optimizations": "gas"
}

FINDING_RE = re.compile(r"\[(H|M|L|N|G)-\d+\]")
FINDING_TITLE_RE = re.compile(
    r"\[\[(?P<id>[A-Z]-\d+)\]\s*(?P<title>.*?)\]"
)
FUNCTION_RE = re.compile(
    r"`([A-Za-z0-9_]+\.[A-Za-z0-9_]+)`"
)
CONTRACT_RE = re.compile(
    r"`([A-Z][A-Za-z0-9_]+)\.sol`"
)
URL_RE = re.compile(r"https?://[^\s)]+")
SYSTEM_PROMPT = """
You are a smart contract security analyst.

Return ONLY valid JSON.
"""

USER_PROMPT = """
Extract structured vulnerability information.

Schema:
{
  "vulnerability_type": "",
  "severity": "",
  "affected_functions": [],
  "affected_contracts": [],
  "root_cause": "",
  "attack_vector": "",
  "prerequisites": [],
  "impact": "",
  "recommendation": "",
  "cwe": "",
  "swc": ""
}

Finding:
...
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": USER_PROMPT
    }
]

In [28]:
def convert_to_html(fp : str) -> list:
    #Convert the text from .md to html
    md = (
        MarkdownIt('commonmark', {'breaks':True,'html':True})
        .use(front_matter_plugin)
        .use(footnote_plugin)
        .enable('table')
    )
    
    with open(fp,"r") as f:
        data = f.read()
    tokens = md.parse(data)
    html_text = md.render(data)
    return tokens, html_text
tokens, html = convert_to_html(fp='3.md')
root = SyntaxTreeNode(tokens=tokens)
def extract_text_sessions(node):
    sessions = []
    current = {
        "title": None,
        "content": []
    }
    for child in node.children:
        if child.type == "heading":
            if current["title"] or current["content"]:
                sessions.append(current)
            title = child.children[0].content

            current = {
                "title": title,
                "content": []
            }
        elif child.type == "paragraph":
            if child.children:
                text = "".join(
                    c.content for c in child.children
                    if hasattr(c, "content")
                )
                current["content"].append(text)
    if current["title"] or current["content"]:
        sessions.append(current)

    return sessions
extract_text_sessions(root)

[{'title': 'Overview', 'content': []},
 {'title': 'About C4',
  'content': ['Code 432n4 (C4) is an open organization that consists of security researchers, auditors, developers, and individuals with domain expertise in the area of smart contracts.',
   'A C4 code contest is an event in which community participants, referred to as Wardens, review, audit, or analyze smart contract logic in exchange for a bounty provided by sponsoring projects.',
   'During the code contest outlined in this document, C4 conducted an analysis of Marginswap’s smart contract system written in Solidity. The code contest took place between April 2 and April 7, 2021.']},
 {'title': 'Wardens',
  'content': ['5 Wardens contributed reports to the Marginswap code contest:',
   'This contest was judged by [Zak Cole](https://twitter.com/0xzak).',
   'Final report assembled by [sockdrawermoney](https://twitter.com/sockdrawermoney).']},
 {'title': 'Summary',
  'content': ['The C4 analysis yielded an aggregated total of

Findings from file

In [29]:
def get_findings_from_sessions(sessions):
    curr_severity = None
    findings = []
    for session in sessions:
        title = session["title"]
        #print(title)
        if title in severity_titles.keys():
            curr_severity = severity_titles[title]
        if FINDING_RE.search(title):
            #print(session["content"])
            match = FINDING_TITLE_RE.search(title)
            if match:
                finding_id = match.group("id")
                finding_title = match.group("title")
                finding = {
                    "severity" : curr_severity,
                    "id" : finding_id,
                    "title" : finding_title,
                    "content" : session["content"]
                }
                findings.append(finding)
    return findings
sessions = extract_text_sessions(root)
get_findings_from_sessions(sessions=sessions)

[{'severity': 'high',
  'id': 'H-01',
  'title': 'Re-entrancy bug allows inflating balance',
  'content': ['One can call the `MarginRouter.crossSwapExactTokensForTokens` function first with a fake contract disguised as a token pair:\n`crossSwapExactTokensForTokens(0.0001 WETH, 0, [ATTACKER_CONTRACT], [WETH, WBTC])`. When the amounts are computed by the `amounts = UniswapStyleLib.getAmountsOut(amountIn - fees, pairs, tokens);` call, the attacker contract returns fake reserves that yield 1 WBTC for the tiny input. The resulting amount is credited through `registerTrade`. Afterwards, `_swapExactT4T([0.0001 WETH, 1 WBTC], 0, [ATTACKER_CONTRACT], [WETH, WBTC])` is called with the fake pair and token amounts. At some point `_swap` is called, the starting balance is stored in `startingBalance`, and the attacker contract call allows a re-entrancy:',
   'From the ATTACKER_CONTRACT we re-enter the `MarginRouter.crossSwapExactTokensForTokens(30 WETH, 0, WETH_WBTC_PAIR, [WETH, WBTC])` function wit

Model finding analysis

In [33]:
def call_llm_agent_api(messages) -> dict:
        try:
            client = OpenAI(
                base_url="http://localhost:11434/v1",
                api_key="ollama",
                timeout=180.0,
                max_retries=5
            )

            completion = client.chat.completions.create(
                model="qwen2.5:1.5b",
                messages=messages,
                #Controls randomness (0 = deterministic)
                temperature=0,
                # Controls token sampling distribution
                top_p=1,
                #Forces valid json output
                response_format={"type": "json_object"}
            )
            #print(completion)

            ans = completion.choices[0].message.content
            return json.loads(ans)

        except openai.APIError as e:
            print(f"OpenAI API returned an API error: {e}")
            raise
        except openai.APIConnectionError as e:
            print(f"Falied to connect to OpenAI API: {e}")
            raise
        except Exception as e:
            print(f"Error at LLM API: {e}")
            raise
def analyse_findings(findings):
    results = list()
    for finding in findings:
        finding_prompt = f"""
        Title: {finding["title"]}

        Description:
        {" ".join(finding["content"])}

        Extract:
        - vulnerability_type
        - affected_functions
        - root_cause
        - impact
        - recommendation
        - vulnerability_type
        - affected_functions
        - affected_contracts
        - attack_vector
        - prerequisites
        - root_cause
        - impact
        - recommendation
        - cwe
        - swc
        """
        #print(finding)
        messages = [
                {"role" : "system",
                "content" : SYSTEM_PROMPT
                },
                {
                    "role" : "user",
                    "content" : finding_prompt
                }
        ]
        try:
                ans = call_llm_agent_api(messages=messages)
                results.append(ans)
        except Exception as e:
                print(f"Error calling llm for finding. Error: {e}")
        
    return results
def build_finding_message(finding):
     finding_prompt = f"""
        Title: {finding["title"]}

        Description:
        {" ".join(finding["content"])}

        Extract:
        - vulnerability_type
        - affected_functions
        - affected_contracts
        - attack_vector
        - prerequisites
        - root_cause
        - impact
        - recommendation
        - cwe
        - swc
        """
     
     return [
             {"role" : "system",
             "content" : SYSTEM_PROMPT
             },
             {
                  "role" : "user",
                  "content" : finding_prompt
             }
        ]


In [34]:

fp = '3.md'
tokens, html = convert_to_html(fp)
root = SyntaxTreeNode(tokens)
sessions = extract_text_sessions(root)
findings = get_findings_from_sessions(sessions)
ft = findings[:2]
res = analyse_findings(ft)
analysis = dict()
analysis["report_fp"] = fp
analysis["analysis_resuls"] = [r for r in res]


In [35]:
analysis["analysis_resuls"]

[{'vulnerability_type': 'Reentrancy',
  'affected_functions': ['MarginRouter.crossSwapExactTokensForTokens'],
  'root_cause': 'The function `crossSwapExactTokensForTokens` allows the attacker to re-enter the same function with fake token amounts, leading to double credit and potential theft of tokens.',
  'impact': 'This vulnerability can allow an attacker to steal multiple times the actual swap result, potentially stealing all available tokens in a controlled manner.',
  'recommendation': 'Add reentrancy guards (from OpenZeppelin) to all external functions of `MarginRouter`.',
  'affected_contracts': ['MarginRouter', 'AttackableContract'],
  'attack_vector': 'First, the attacker disguises their contract as a token pair and calls `crossSwapExactTokensForTokens` with fake amounts. Then, they re-enter the function with actual amounts.',
  'prerequisites': 'The attacker must have access to the `MarginRouter` contract and control over the `AttackableContract`.',
  'cwe': 'CWE-259',
  'swc'

In [ ]:
curr = os.getcwd()
